# Watching a simulation run

`newen.education.view()` draws the atoms of a running simulation as lit spheres, next to two
strip charts: kinetic, potential and total energy, and temperature. It needs the `education`
extra (`anywidget`) and works without an internet connection: the 3D drawing code ships with
the package. By default three.js draws glossy spheres with soft shadows; `renderer="3dmol"`
switches to 3Dmol.js.

The input files come from the repository's reference suite. The paths assume Jupyter was
started in `py-newen/notebooks/`.

In [ ]:
from pathlib import Path

from newen import Recipe, Simulation, System
from newen.education import lj_gas, record, view, with_bath

REF_DIR = Path("../../crates/newen-md/tests/gromosXX_references")

## 216 water molecules, live

`view(sim)` shows the widget straight away, steps the simulation, and sends each sample to it
as it arrives (here 100 samples, one every 5 steps). When the run is over the widget replays
the whole run on its own: use **Pause** and the slider to step through it, drag to rotate,
scroll to zoom.

A `Simulation` does not know its atom names, so `system=` passes them in: they give the
colours (O red, H white) and the sphere sizes.

In [ ]:
water = System.from_files(
    str(REF_DIR / "water_216_box" / "water_216_box.topo"),
    str(REF_DIR / "water_216_box" / "water_216_box.conf"),
)
sim = Simulation(water, Recipe.from_imd(str(REF_DIR / "water_216_box" / "water_216_box.in")))
view(sim, frames=100, every=5, system=water)

**What to look for.** The total energy (dark line) stays almost flat while kinetic and
potential energy trade places: energy moves between motion and interaction, it is not made or
lost. The temperature is the kinetic energy in other units, so its chart has the same shape
as the kinetic line. Watch the molecules: each water keeps its shape, because its bonds are
held rigid, while the molecules jostle each other.

## Two argon atoms, recorded first

`record()` runs a simulation without drawing anything and keeps what it produced; `view()` of
that recording just plays it. A recording can also be saved: `rec.save_pdb("run.pdb")` writes
every frame for any molecular viewer.

In [ ]:
argon = System.from_files(
    str(REF_DIR / "pair_lj" / "pair_lj.topo"), str(REF_DIR / "pair_lj" / "pair_lj.conf")
)
recipe = Recipe.from_imd(str(REF_DIR / "pair_lj" / "pair_lj.in"))
rec = record(Simulation(argon, recipe), frames=200, every=5, system=argon)
print(rec.frames.shape, "frames x atoms x xyz (nm)")
view(rec)

The two atoms start at rest, 0.35 nm apart: they attract, fall together, bounce off each
other's repulsive wall and fly apart again. Kinetic energy peaks when they are closest to the
bottom of the Lennard-Jones well; the potential energy mirrors it, and the total stays put.

3Dmol.js: ambient occlusion and outlines, the molecular look.

In [ ]:
view(rec, renderer="3dmol")

## A 2D gas, coloured by speed

`lj_gas(dim=2)` keeps every atom in one plane, so the viewer lays the gas on a table.
`colors="speed"` colours each atom by its speed, slow deep blue to fast pale gold, with the
scale in the corner. The gas starts on a square lattice with random velocities; watch it melt.
Each atom changes colour at every collision, while fast and slow atoms stay mixed everywhere.

In [ ]:
view(Simulation(*lj_gas(n=64, dim=2)), frames=150, every=10, colors="speed")

## A hot gas meets a cold bath

The same gas starts hot (reduced temperature 2, about 240 K), and `with_bath` couples it to a
Berendsen heat bath at 120 K. The colours are speeds: watch them sink as Berendsen pulls the
gas down to 120 K.

In [ ]:
system, recipe = lj_gas(n=64, dim=2, temperature=2.0)
sim = Simulation(system, with_bath(recipe, temperature_K=300.0, tau=0.5))
view(sim, frames=300, every=10, colors="speed")